# Recomendacion de Productos - Olist

Analisis completo del caso 2 del proyecto: segmentacion de productos con K-Means,
analisis de preferencias de clientes y generacion de recomendaciones basadas en clusters.

## 1. Configuracion del entorno

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.load_data import load_csv
from src.data.preprocess import build_customer_product_history, build_product_dataset
from src.features.feature_engineering import create_recommendation_features
from src.pipelines.recommendation_pipeline import run_recommendation_pipeline
from src.recommendation.clustering import build_cluster_summary, build_pca_projection, find_optimal_k
from src.recommendation.engine import analyze_customer_preferences, generate_sample_recommendations

%matplotlib inline
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 60)

## 2. Analisis Exploratorio de Datos (EDA)

Primero cargamos los datasets raw mas relevantes para entender el universo de productos,
ordenes y reviews que alimentan el pipeline de recomendacion.

In [ ]:
products = load_csv("olist_products_dataset.csv")
orders = load_csv("olist_orders_dataset.csv")
order_items = load_csv("olist_order_items_dataset.csv")
reviews = load_csv("olist_order_reviews_dataset.csv")

print("Dimensiones de los datasets:")
for name, df in [
    ("Products", products),
    ("Orders", orders),
    ("Order Items", order_items),
    ("Reviews", reviews),
]:
    print(f"  {name}: {df.shape[0]} filas x {df.shape[1]} columnas")

print(f"\nOrdenes entregadas: {(orders['order_status'] == 'delivered').sum()}")

In [ ]:
product_data = build_product_dataset()
print(f"Dataset de productos: {product_data.shape[0]} filas x {product_data.shape[1]} columnas")
print(f"Categorias unicas: {product_data['category'].nunique()}")
product_data.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

product_data['category'].value_counts().head(12).plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Top 12 categorias por cantidad de productos')
axes[0].set_ylabel('Cantidad de productos')

product_data['avg_review_score'].plot(kind='hist', bins=20, ax=axes[1], color='coral')
axes[1].set_title('Distribucion del review score promedio')
axes[1].set_xlabel('Review score promedio')

plt.tight_layout()
plt.show()

## 3. Clustering de Productos con K-Means

Se crean features combinando categoria, precio, reviews y popularidad del producto.
Luego se busca el mejor numero de clusters usando silhouette score.

In [ ]:
clustering_features, feature_names = create_recommendation_features(product_data)
print(f"Features de clustering: {clustering_features.shape[1]}")
print(f"Primeras 15 features: {feature_names[:15]}")

best_k, silhouette_scores, kmeans_model = find_optimal_k(clustering_features)
product_data['cluster'] = kmeans_model.labels_

cluster_summary = build_cluster_summary(product_data)
cluster_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(list(silhouette_scores.keys()), list(silhouette_scores.values()), marker='o', color='steelblue')
ax.axvline(best_k, color='firebrick', linestyle='--', label=f'Mejor k = {best_k}')
ax.set_title('Seleccion de k con silhouette score')
ax.set_xlabel('Numero de clusters')
ax.set_ylabel('Silhouette score')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Visualizacion de Clusters con PCA

Aplicamos PCA para proyectar los clusters de productos en 2D y 3D y asi interpretar mejor
la separacion entre segmentos.

In [ ]:
pca_2d, pca_model_2d = build_pca_projection(clustering_features, kmeans_model.labels_, n_components=2)
pca_3d, pca_model_3d = build_pca_projection(clustering_features, kmeans_model.labels_, n_components=3)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(pca_2d['PC1'], pca_2d['PC2'], c=pca_2d['cluster'], cmap='viridis', alpha=0.55, s=14)
fig.colorbar(scatter, label='Cluster')
ax.set_title('Clusters de productos en PCA 2D')
ax.set_xlabel(f"PC1 ({pca_model_2d.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca_model_2d.explained_variance_ratio_[1]:.1%})")
plt.tight_layout()
plt.show()

print(f"Varianza explicada PCA 2D: {pca_model_2d.explained_variance_ratio_.sum():.2%}")
print(f"Varianza explicada PCA 3D: {pca_model_3d.explained_variance_ratio_.sum():.2%}")

In [ ]:
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
scatter = ax.scatter(
    pca_3d['PC1'],
    pca_3d['PC2'],
    pca_3d['PC3'],
    c=pca_3d['cluster'],
    cmap='viridis',
    alpha=0.45,
    s=12,
)
fig.colorbar(scatter, ax=ax, pad=0.12, label='Cluster')
ax.set_title('Clusters de productos en PCA 3D')
ax.set_xlabel(f"PC1 ({pca_model_3d.explained_variance_ratio_[0]:.1%})")
ax.set_ylabel(f"PC2 ({pca_model_3d.explained_variance_ratio_[1]:.1%})")
ax.set_zlabel(f"PC3 ({pca_model_3d.explained_variance_ratio_[2]:.1%})")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

product_data['cluster'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title('Cantidad de productos por cluster')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Cantidad de productos')

top_categories = product_data['category'].value_counts().head(15).index
heatmap_data = pd.crosstab(product_data.loc[product_data['category'].isin(top_categories), 'category'], product_data['cluster'])
heatmap_data = heatmap_data.div(heatmap_data.sum(axis=1), axis=0)
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[1])
axes[1].set_title('Heatmap de categorias por cluster (top 15)')
axes[1].set_xlabel('Cluster')
axes[1].set_ylabel('Categoria')

plt.tight_layout()
plt.show()

## 5. Preferencias de Clientes y Recomendaciones

A partir del historial de compras se calculan preferencias por cluster y categoria,
y luego se generan recomendaciones priorizando afinidad, review score y popularidad.

In [ ]:
history = build_customer_product_history()
history_enriched, cluster_preferences, category_preferences, customer_summary = analyze_customer_preferences(history, product_data)

print(f"Historial enriquecido: {history_enriched.shape[0]} filas")
print(f"Clientes analizados: {customer_summary.shape[0]}")
customer_summary.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

cluster_preferences['cluster'].value_counts().sort_index().plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Clusters presentes en preferencias de clientes')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Cantidad de registros de preferencia')

customer_summary['preferred_category'].value_counts().head(12).plot(kind='bar', ax=axes[1], color='coral')
axes[1].set_title('Top categorias preferidas por clientes')
axes[1].set_ylabel('Cantidad de clientes')

plt.tight_layout()
plt.show()

In [ ]:
sample_recommendations = generate_sample_recommendations(
    history_enriched,
    product_data,
    cluster_preferences,
    category_preferences,
    top_customers=10,
    top_n=5,
)

print(f"Recomendaciones generadas: {sample_recommendations.shape[0]}")
sample_recommendations.head(15)

## 6. Ejecucion End-to-End del Pipeline

Esta celda ejecuta todo el flujo productivo y deja guardados los artefactos en `data/processed/`,
`reports/results/` y `reports/figures/`.

In [ ]:
product_clusters, customer_cluster_preferences, sample_recommendations = run_recommendation_pipeline()

## 7. Conclusiones

- Los productos quedan agrupados considerando categoria, precio, reviews y demanda.
- El historial de compras permite inferir clusters y categorias preferidas por cada cliente.
- Las recomendaciones finales priorizan productos no comprados previamente con mayor afinidad al perfil del cliente.
- PCA facilita interpretar visualmente la separacion de clusters en 2D y 3D.